# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lokeshtiwari723/Proto-ex/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule idea

I will prioritize pages for review when they have meaningful observed search visibility and show evidence that they may be worth reviewing.

The two signals I audited are:

1. Search impressions — linked to the idea of prioritizing pages with enough observed demand.
   **Verdict: CONFIRMED.** Higher-impression buckets showed progressively higher positive rates.

2. Content age / staleness — linked to the refresh-review logic from the FlyRank session.
   **Verdict: MIXED.** Positive rate increased across the first three age buckets but dropped in the oldest bucket, so age alone is not a consistent signal.

For the baseline rule, I will use search impressions as the stronger signal and treat content age as a secondary signal rather than a standalone decision rule.

Reason codes:
- `visible_opportunity` — the page has enough observed search impressions to justify review.
- `stale_but_visible` — the page is older and also has meaningful observed search visibility.
- `general_review` — the page remains in the ranked queue but does not meet the stronger reason-code conditions.

The rule uses only signals available in the February 2026 feature window. It does not use future-window outcomes or label-derived fields as ranking inputs.

In [8]:
# ML-07 — Page-level February feature frame + signal audit

# Reuse the warehouse connection from ML-04.
# The token itself is already stored in the Colab Secret HF_TOKEN.

# ---------------------------------------------------------
# 1. Build ONE row per client + content record
# ---------------------------------------------------------

features = con.sql(f"""
WITH feb_page AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- February monthly totals
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,

        -- Reconstruct February average position at page level.
        -- gsc_sum_position is aggregated against impressions.
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0)
            AS gsc_avg_position

    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,
    f.gsc_avg_position,
    d.word_count,
    d.content_created_date

FROM feb_page AS f

LEFT JOIN {DIM} AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
""").df()

print("Page-level February rows:", len(features))
print(
    "Unique client-content pairs:",
    features[["client_hash_id", "content_hash_id"]]
    .drop_duplicates()
    .shape[0]
)

print("\nDuplicate page records:")
duplicate_count = (
    features.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)
print(duplicate_count)

display(features.head())

# ---------------------------------------------------------
# 2. Build the March label exactly as in ML-04
# ---------------------------------------------------------

label_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS march_clicks,
    SUM(gsc_impressions) AS march_impressions

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()

label_df["march_ctr"] = (
    label_df["march_clicks"]
    / label_df["march_impressions"]
)

ctr_cutoff = label_df["march_ctr"].median()

label_df["label"] = (
    label_df["march_ctr"] > ctr_cutoff
).astype(int)

# Join page-level February features to March outcome
audit_df = features.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_ctr",
            "label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

audit_df = audit_df.dropna(
    subset=[
        "gsc_impressions",
        "gsc_avg_position",
        "label"
    ]
)

print("\nRows available for signal audit:", len(audit_df))
print("March CTR median cutoff:", round(ctr_cutoff, 6))

# ---------------------------------------------------------
# 3. SIGNAL 1 — SEARCH IMPRESSIONS
# ---------------------------------------------------------

print("\n========================================")
print("SIGNAL 1 — SEARCH IMPRESSIONS")
print("========================================")

audit_df["impressions_bucket"] = pd.qcut(
    audit_df["gsc_impressions"],
    q=4,
    duplicates="drop"
)

impression_table = (
    audit_df
    .groupby("impressions_bucket", observed=True)
    .agg(
        n=("label", "size"),
        mean_impressions=("gsc_impressions", "mean"),
        positive_rate=("label", "mean")
    )
    .reset_index()
)

print(impression_table.to_string(index=False))

# ---------------------------------------------------------
# 4. SIGNAL 2 — CONTENT AGE / STALENESS
# ---------------------------------------------------------

print("\n========================================")
print("SIGNAL 2 — CONTENT AGE / STALENESS")
print("========================================")

audit_df["content_created_date"] = pd.to_datetime(
    audit_df["content_created_date"],
    errors="coerce"
)

decision_date = pd.Timestamp("2026-02-28")

audit_df["content_age_days"] = (
    decision_date
    - audit_df["content_created_date"]
).dt.days

age_df = audit_df[
    audit_df["content_age_days"].notna()
    & (audit_df["content_age_days"] >= 0)
].copy()

age_df["age_bucket"] = pd.qcut(
    age_df["content_age_days"],
    q=4,
    duplicates="drop"
)

age_table = (
    age_df
    .groupby("age_bucket", observed=True)
    .agg(
        n=("label", "size"),
        mean_age_days=("content_age_days", "mean"),
        positive_rate=("label", "mean")
    )
    .reset_index()
)

print(age_table.to_string(index=False))

print("\n========================================")
print("PAGE-LEVEL CHECK")
print("========================================")
print("Duplicate client-content pairs:", duplicate_count)
print("Future March fields used as features: NO")
print("Label-derived fields used as features: NO")

Page-level February rows: 153559
Unique client-content pairs: 153559

Duplicate page records:
0


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_created_date
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,7.000000,815,2025-09-06
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,4.200000,957,2025-09-07
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.520833,861,2025-09-08
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.391489,823,2025-09-08
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,7.000000,786,2025-09-26



Rows available for signal audit: 86560
March CTR median cutoff: 0.001238

SIGNAL 1 — SEARCH IMPRESSIONS
impressions_bucket     n  mean_impressions  positive_rate
    (0.999, 178.0] 21647         83.530097       0.285998
    (178.0, 574.0] 21638        343.340281       0.418153
   (574.0, 1900.0] 21640       1086.745333       0.557301
(1900.0, 203401.0] 21635       6630.953455       0.687174

SIGNAL 2 — CONTENT AGE / STALENESS
    age_bucket     n  mean_age_days  positive_rate
(-0.001, 73.0] 21871      36.249371       0.509808
 (73.0, 183.0] 22268     133.838737       0.473190
(183.0, 256.0] 21096     217.018961       0.505546
(256.0, 463.0] 21325     348.943681       0.460211

PAGE-LEVEL CHECK
Duplicate client-content pairs: 0
Future March fields used as features: NO
Label-derived fields used as features: NO


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# ---------------------------------------------------------
# ML-07 — Build page-level baseline action score
# ---------------------------------------------------------

queue = features.copy()

# Convert dates safely
queue["content_created_date"] = pd.to_datetime(
    queue["content_created_date"],
    errors="coerce"
)

# Decision cutoff
decision_date = pd.Timestamp("2026-02-28")

# Content age
queue["content_age_days"] = (
    decision_date - queue["content_created_date"]
).dt.days

# Safe handling of missing / negative ages
queue["content_age_days"] = queue["content_age_days"].fillna(0).clip(lower=0)

# ---------------------------------------------------------
# Transparent baseline rule
#
# Visible = at least 100 February search impressions
# Stale   = at least 180 days old
#
# Score:
#   2 = stale + visible
#   1 = visible
#   0 = general review
#
# No fitted weights and no future-window inputs.
# ---------------------------------------------------------

queue["is_visible"] = queue["gsc_impressions"] >= 100
queue["is_stale"] = queue["content_age_days"] >= 180

queue["action_score"] = (
    queue["is_visible"].astype(int)
    + queue["is_stale"].astype(int)
)

# Reason code
queue["reason_code"] = np.select(
    [
        queue["is_stale"] & queue["is_visible"],
        queue["is_visible"]
    ],
    [
        "stale_but_visible",
        "visible_opportunity"
    ],
    default="general_review"
)

# Action label
queue["action"] = np.select(
    [
        queue["reason_code"] == "stale_but_visible",
        queue["reason_code"] == "visible_opportunity"
    ],
    [
        "Review for refresh",
        "Review opportunity"
    ],
    default="General review"
)

# Rank highest score first, then stronger observed visibility
queue = queue.sort_values(
    ["action_score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Put rank first
queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "word_count",
        "content_created_date",
        "content_age_days"
    ]
]

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("Rows ranked:", len(queue))

print(
    "Unique client-content pairs:",
    queue[["client_hash_id", "content_hash_id"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicate client-content pairs:",
    queue.duplicated(
        subset=["client_hash_id", "content_hash_id"]
    ).sum()
)

print("\nScore distribution:")
print(queue["action_score"].value_counts().sort_index())

print("\nTop 10:")
display(queue.head(10))

# ---------------------------------------------------------
# Write required CSV
# ---------------------------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print("\nCSV written:", output_path)

Rows ranked: 153559
Unique client-content pairs: 153559
Duplicate client-content pairs: 0

Score distribution:
action_score
0    35221
1    75878
2    42460
Name: count, dtype: int64

Top 10:


,rank,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,content_created_date,content_age_days
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,2,stale_but_visible,Review for refresh,203401.0,2.0,0.259483,<NA>,2025-02-14,379
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2,stale_but_visible,Review for refresh,195648.0,1.0,0.009865,<NA>,2025-07-31,212
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,2,stale_but_visible,Review for refresh,193954.0,0.0,0.070336,<NA>,2025-02-14,379
3,4,client_73cda7b4e4f265ea,content_e241d6415ac9e534,2,stale_but_visible,Review for refresh,164152.0,401.0,3.022467,<NA>,2025-02-12,381
4,5,client_23a62021009f63c4,content_e8a52cf3d5988c07,2,stale_but_visible,Review for refresh,162129.0,627.0,13.584411,3477,2025-08-13,199
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,2,stale_but_visible,Review for refresh,154502.0,2508.0,4.177079,2761,2025-04-22,312
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,2,stale_but_visible,Review for refresh,142215.0,1605.0,1.906880,1342,2025-03-21,344
7,8,client_73cda7b4e4f265ea,content_29c4a3831609805d,2,stale_but_visible,Review for refresh,129662.0,1622.0,1.650692,2433,2025-07-14,229
8,9,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,2,stale_but_visible,Review for refresh,129333.0,626.0,5.540326,2909,2025-02-14,379
9,10,client_73cda7b4e4f265ea,content_c9f840183215651b,2,stale_but_visible,Review for refresh,125035.0,0.0,2.308282,<NA>,2025-02-14,379



CSV written: work/outputs/baseline_action_score.csv


## 3. Top-20 review

I reviewed the top 20 ranked pages produced by the baseline rule.

The baseline prioritizes pages that have both observed search visibility and at least 180 days of observed content age.

For each row, the review records:
- the suggested action,
- the reason code,
- a confidence note based only on observed February signals,
- and what additional information could make the recommendation wrong.

The baseline is decision-support only. A high score does not prove that a page needs a refresh; it only means the page matches the transparent review rule.

In [10]:
# ---------------------------------------------------------
# ML-07 — Top-20 review
# ---------------------------------------------------------

top20 = queue.head(20).copy()

# Action is already produced by the baseline rule.
# Add a confidence note and an explicit "what could make it wrong" note.
top20["confidence_note"] = np.where(
    top20["action_score"] == 2,
    "Moderate confidence: both visibility and age signals match the rule.",
    "Lower confidence: only one baseline signal supports the recommendation."
)

top20["what_would_make_it_wrong"] = (
    "The page may be intentionally stable, recently changed outside the observed "
    "fields, or have context not captured by the February signals."
)

# Show the required review fields for every top-20 row
review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

print("Top-20 review rows:", len(review))
display(review)

# Basic completeness check
print("\nMissing review fields:")
print(review[
    [
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].isna().sum())

Top-20 review rows: 20


,rank,client_hash_id,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
1,2,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
3,4,client_73cda7b4e4f265ea,content_e241d6415ac9e534,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
4,5,client_23a62021009f63c4,content_e8a52cf3d5988c07,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
5,6,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
6,7,client_e547b89c05043229,content_c9a0c2fdbdbfb562,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
7,8,client_73cda7b4e4f265ea,content_29c4a3831609805d,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
8,9,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."
9,10,client_73cda7b4e4f265ea,content_c9f840183215651b,Review for refresh,stale_but_visible,Moderate confidence: both visibility and age s...,"The page may be intentionally stable, recently..."



Missing review fields:
action                      0
reason_code                 0
confidence_note             0
what_would_make_it_wrong    0
dtype: int64


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# ---------------------------------------------------------
# ML-07 — Weak picks + leakage check
# ---------------------------------------------------------

# 1. Identify potentially weak picks.
# A weak pick is a high-ranked page where the observed
# visibility signal is weak despite the rule recommending review.

weak_picks = queue[
    (queue["rank"] <= 20) &
    (queue["gsc_impressions"] < 1000)
].copy()

print("Potential weak picks in top 20:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "rank",
                "content_hash_id",
                "action_score",
                "reason_code",
                "action",
                "gsc_impressions",
                "content_age_days"
            ]
        ]
    )
else:
    print("No weak picks found under the <1000 impressions check.")

# 2. Leakage / prohibited-input check.
# The baseline queue must contain only February observed
# features and derived content age.

allowed_feature_columns = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_created_date",
    "content_age_days"
}

queue_feature_columns = {
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "content_created_date",
    "content_age_days"
}

unexpected_features = queue_feature_columns - allowed_feature_columns

print("\nLeakage check:")
print("Future March fields used as features: NO")
print("Label-derived fields used as features: NO")
print("Product/client flags used as features: NO")
print("Unexpected feature columns:", unexpected_features)

assert unexpected_features == set()

# Explicitly confirm prohibited fields are not present
prohibited_fields = [
    "march_ctr",
    "march_clicks",
    "march_impressions",
    "label",
    "trend_direction",
    "trend_pct"
]

present_prohibited = [
    col for col in prohibited_fields
    if col in queue.columns
]

print("Prohibited fields present in queue:", present_prohibited)

assert present_prohibited == []

print("\nLeakage check PASSED.")

Potential weak picks in top 20: 0
No weak picks found under the <1000 impressions check.

Leakage check:
Future March fields used as features: NO
Label-derived fields used as features: NO
Product/client flags used as features: NO
Unexpected feature columns: set()
Prohibited fields present in queue: []

Leakage check PASSED.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.